# 📊 TD07 — Analytics Module & Quantitative Indicators
## MarketPulse — Feature 1: `feature/analytics` ➔ `src/analytics.py`

Welcome to the first MarketPulse practical notebook!

---
### 👥 Team Organization (4-Student Group)
* **Assigned Developer (PR Author)**: **Student 1** (Lead Analytics)
* **Official Code Reviewer**: **Student 2**
* **Dedicated Git Branch**: `feature/analytics`
* **Assessment Milestone**: Contributes to **Checkpoint B (30%)**
---

### 🎯 Notebook Objectives:
1. Understand and implement the core financial functions required by the **MarketPulse Analytical Contract**:
   - Date indexing and temporal alignment of two time series.
   - Calculation of **period return**.
   - **Base-100 normalization**.
   - Calculation of **relative performance** (outperformance / underperformance).
2. Test and validate these functions interactively using project sample data.
3. Follow the **Git Integration Guide** at the end of this notebook to export this clean code into **`src/analytics.py`** via a Pull Request.

---
### 📐 Functional Contract
As a reminder, the MarketPulse analytical contract specifies:
- **Period return**: $\left(\frac{P_{\text{end}}}{P_{\text{start}}} - 1.0\right) \times 100.0$
- **Base-100 index**: $\left(\frac{P_t}{P_0}\right) \times 100.0$
- **Relative performance**: $R_{\text{instrument}} - R_{\text{benchmark}}$ (in percentage points)


## 1. Date Indexing and Temporal Alignment


In [ ]:
def index_by_date(prices):
    """
    Index a list of canonical observation dictionaries by date ('YYYY-MM-DD').
    
    Args:
        prices (list): List of canonical observation dictionaries.
        
    Returns:
        dict: Mapping of {date: observation_dict}.
    """
    return {row["date"]: row for row in prices}


def align_series(instrument_prices, benchmark_prices):
    """
    Align two price series on the sorted intersection of their common dates.
    
    Args:
        instrument_prices (list): Observations of the primary asset.
        benchmark_prices (list): Observations of the benchmark index.
        
    Returns:
        tuple: (aligned_instrument_prices, aligned_benchmark_prices)
    """
    instrument_by_date = index_by_date(instrument_prices)
    benchmark_by_date = index_by_date(benchmark_prices)

    common_dates = sorted(set(instrument_by_date.keys()) & set(benchmark_by_date.keys()))

    aligned_instrument = [instrument_by_date[date] for date in common_dates]
    aligned_benchmark = [benchmark_by_date[date] for date in common_dates]

    return aligned_instrument, aligned_benchmark


## 2. Standard Quantitative Calculations


In [ ]:
def calculate_period_return(prices):
    """
    Calculate total arithmetic return over the observed period:
    Formula: ((last_close / first_close) - 1.0) * 100.0
    
    Args:
        prices (list): Chronologically sorted list of observations.
        
    Returns:
        float: Percentage return (e.g., 6.48 for +6.48%).
    """
    if not prices:
        raise ValueError("Cannot compute return on an empty series.")
    
    first_close = float(prices[0]["close"])
    last_close = float(prices[-1]["close"])

    if first_close == 0.0:
        raise ZeroDivisionError("Initial price cannot be zero.")

    return ((last_close / first_close) - 1.0) * 100.0


def calculate_base_100(prices):
    """
    Normalize a price series to Base 100 (where day zero equals 100.0).
    Formula: (current_close / first_close) * 100.0
    
    Args:
        prices (list): List of canonical observations.
        
    Returns:
        list: List of dictionaries [{'date': ..., 'ticker': ..., 'value': float}, ...].
    """
    if not prices:
        raise ValueError("Cannot calculate Base 100 on an empty series.")

    first_close = float(prices[0]["close"])
    if first_close == 0.0:
        raise ZeroDivisionError("Initial price cannot be zero.")

    result = []
    for row in prices:
        result.append({
            "date": row["date"],
            "ticker": row["ticker"],
            "value": (float(row["close"]) / first_close) * 100.0,
        })
    return result


def calculate_relative_performance(instrument_return, benchmark_return):
    """
    Calculate relative outperformance or underperformance in percentage points.
    Formula: instrument_return - benchmark_return
    
    Args:
        instrument_return (float): Asset return in %.
        benchmark_return (float): Benchmark return in %.
        
    Returns:
        float: Difference in percentage points (pts).
    """
    return instrument_return - benchmark_return


## 3. Interactive Validation with Project Sample Data


In [ ]:
import csv
from pathlib import Path

# Load sample prices from data/sample/prices.csv
sample_prices_path = Path("../data/sample/prices.csv")
if not sample_prices_path.exists():
    sample_prices_path = Path("data/sample/prices.csv")

with open(sample_prices_path, encoding="utf-8") as f:
    all_prices = list(csv.DictReader(f))

# Separate AAPL and SP500 series
aapl_prices = [row for row in all_prices if row["ticker"] == "AAPL"]
sp500_prices = [row for row in all_prices if row["ticker"] == "SP500"]

print(f"Raw AAPL observations   : {len(aapl_prices)}")
print(f"Raw SP500 observations  : {len(sp500_prices)}")

# 1. Alignment
aligned_aapl, aligned_sp500 = align_series(aapl_prices, sp500_prices)
print(f"Common aligned dates    : {len(aligned_aapl)}")

# 2. Returns
aapl_return = calculate_period_return(aligned_aapl)
sp500_return = calculate_period_return(aligned_sp500)
rel_perf = calculate_relative_performance(aapl_return, sp500_return)

print(f"AAPL period return      : {aapl_return:+.2f}%")
print(f"SP500 period return     : {sp500_return:+.2f}%")
print(f"Relative performance    : {rel_perf:+.2f} percentage points")

# 3. Base 100
aapl_base_100 = calculate_base_100(aligned_aapl)
sp500_base_100 = calculate_base_100(aligned_sp500)

print(f"AAPL Base 100 start     : {aapl_base_100[0]['value']:.1f} | end: {aapl_base_100[-1]['value']:.2f}")
print(f"SP500 Base 100 start    : {sp500_base_100[0]['value']:.1f} | end: {sp500_base_100[-1]['value']:.2f}")


---
## 🛠️ Step-by-Step Git Integration Guide (Team of 4)

Follow these steps to turn your validated notebook code into a production module:

### Step 1: Branch Preparation (by Student 1)
In your local terminal:
```bash
# 1. Switch to main and pull latest team work
git switch main
git pull origin main

# 2. Create and switch to the feature branch
git switch -c feature/analytics
```

### Step 2: Create the `src/analytics.py` Module
1. Create or edit **`src/analytics.py`**.
2. Copy the 5 validated pure functions:
   - `index_by_date`
   - `align_series`
   - `calculate_period_return`
   - `calculate_base_100`
   - `calculate_relative_performance`
3. ⚠️ **Golden Rule**: Do not include temporary print statements or sample loading code in `src/analytics.py`. The module must remain pure and reusable.

### Step 3: Local Verification
Run from the repository root:
```bash
python3 -c "import src.analytics as a; print('✅ analytics.py successfully imported!')"
```

### Step 4: Commit and Push to GitHub
```bash
git add src/analytics.py
git commit -m "feat(analytics): implement period return, base-100 and series alignment"
git push -u origin feature/analytics
```

### Step 5: Open the Pull Request on GitHub
- Go to your team repository on GitHub (`<team-owner>/esilv-marketpulse-gXX-tYY`).
- Click **Compare & pull request**:
  - **Base branch**: `main`
  - **Compare branch**: `feature/analytics`
  - **Title**: `feat: add core quantitative analytics module`
  - **Assignee**: Student 1
  - **Reviewer**: Student 2
- 📸 **Checkpoint B Evidence (Student 1)**: Capture a screenshot of this PR for [`01_pull_request.png`](../docs/04_CHECKPOINTS_AND_EVIDENCE.md).

### Step 6: Code Review & Merge (by Student 2)
1. Student 2 navigates to the PR on GitHub and clicks **Files changed**.
2. Verify that only pure functions are present, leave a constructive comment.
3. 📸 **Checkpoint B Evidence (Student 2)**: Capture a screenshot of your review for [`02_code_review.png`](../docs/04_CHECKPOINTS_AND_EVIDENCE.md).
4. Student 2 approves (**Approve**) and clicks **Merge pull request**.
5. All team members can now sync their local `main` branch (`git switch main && git pull`).
